# Phase 5: Risk Model

This notebook implements a Decision Tree Classifier using `scikit-learn` to classify projects into High, Medium, or Low cost-overrun risk categories.

**Features used:**
- `planned_duration`, `avg_wastage_pct`, `avg_attendance_pct`
- `attendance_trend_4wk` (rolling 4-week attendance slope)
- `vendor_delay_std` (delivery delay variability)

SHAP values are computed at prediction time to surface the top-2 risk drivers per project.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Append src folder to path
sys.path.append(os.path.abspath('../src'))

import db_connection
import risk_model

## 1. Prepare Features

In [ ]:
conn = db_connection.get_connection()
df_features = risk_model.prepare_features(conn)
conn.close()

df_features.head()

## 2. Train Model and Predict Risk

In [ ]:
df_results, model, importances, encoded_feature_cols, X_all = risk_model.train_and_predict(df_features)

print("Feature Importances:")
for feat, imp in sorted(importances.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {feat}: {imp:.4f}")

print("\nPredicted risk distribution:")
print(df_results["predicted_risk_label"].value_counts())

## 3. Visualize Feature Importances

In [ ]:
sns.set_theme(style="whitegrid")
df_imp = pd.DataFrame(list(importances.items()), columns=["Feature", "Importance"]).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_imp, x="Importance", y="Feature", palette="mako")
plt.title("Risk Model Feature Importances (Decision Tree)", fontsize=14, fontweight='bold')
plt.xlabel("Feature Importance", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/feature_importances.png", dpi=300)
plt.show()

In [ ]:
df_reasons = risk_model.get_top_shap_reasons(
    model, X_all, encoded_feature_cols, df_results, top_n=2
)

merged = df_results[["project_id", "project_name", "predicted_risk_label"]].merge(
    df_reasons, on="project_id"
)
print("High-Risk projects with SHAP reasons:")
print(merged[merged["predicted_risk_label"] == "High"].to_string(index=False))

## 5. Export Risk Flags Report

Exports project-level risk predictions together with per-project SHAP top-2 reasons.

In [ ]:
os.makedirs("../reports", exist_ok=True)
output_file = "../reports/risk_flags.csv"

df_reasons = risk_model.get_top_shap_reasons(
    model, X_all, encoded_feature_cols, df_results, top_n=2
)

df_output = df_results[
    ["project_id", "project_name", "project_type", "status", "predicted_risk_label"]
].merge(df_reasons, on="project_id", how="left")

top_features = sorted(importances.items(), key=lambda x: x[1], reverse=True)[:3]
note = (
    "# RISK MODEL REPORT: Gupta Engineers & Contractors\n"
    "# Classifies projects into High / Medium / Low cost-overrun risk.\n"
    f"# Top 3 global features: 1. {top_features[0][0]} ({top_features[0][1]:.2f}), "
    f"2. {top_features[1][0]} ({top_features[1][1]:.2f}), "
    f"3. {top_features[2][0]} ({top_features[2][1]:.2f})\n"
    "# top_reason_1 / top_reason_2 = per-project SHAP drivers.\n"
    "#\n"
)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(note)
    df_output.to_csv(f, index=False)

print(f"Risk flags exported to {output_file}")
df_output.head(10)